# Notebook 03 — RQ1: Signal Predictability

Fama-MacBeth cross-sectional alpha test, XGBoost walk-forward, and SHAP feature importance. Tests H0: net alpha = 0 after transaction costs.

**QM640 Data Analytics Capstone · Walsh College · Anupam K Mitra · May 2026**

---
> **Standalone notebook** — all code is self-contained. Run cells top-to-bottom. Data is downloaded automatically on first run.

## 1. Setup & Data

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
warnings.filterwarnings('ignore')
%matplotlib inline

# ── Config ────────────────────────────────────────────────────
TICKERS    = ['SPY','QQQ','IWM','GLD','TLT','HYG','IEF',
               'USO','XLF','XLE','XLK','BTC-USD']
RF_ANNUAL  = 0.04
COST_BPS   = 5.0
START, END = '2010-01-01', '2024-12-31'
ALPHA_HLZ  = 3.0   # Harvey, Liu & Zhu (2016) t-threshold
SIGNAL_HOR = 5     # 5-day forward return

import yfinance as yf
raw    = yf.download(TICKERS, start=START, end=END,
                     auto_adjust=True, progress=False)
prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
prices = prices.ffill(limit=3).replace(0, np.nan)
log_ret= np.log(prices / prices.shift(1)).clip(-0.20, 0.20)
print(f'Loaded: {prices.shape}')

## 2. Feature Engineering — 5 Signal Families

In [ ]:
cost   = COST_BPS / 10_000
rf_d   = RF_ANNUAL / 252

def build_features(p, r):
    f = pd.DataFrame(index=p.index)
    # Momentum family
    for d in [5, 21, 63, 126, 252]:
        f[f'mom_{d}d'] = np.log(p / p.shift(d)).shift(1)
    # Mean-reversion
    f['rev_1d'] = r.shift(1)
    f['rev_5d'] = r.rolling(5).sum().shift(1)
    # RSI
    g  = r.clip(lower=0).rolling(14).mean()
    l_ = (-r.clip(upper=0)).rolling(14).mean()
    f['rsi'] = (100 - 100 / (1 + g / (l_ + 1e-9))).shift(1)
    # Volatility
    f['vol_21']   = r.rolling(21).std().shift(1) * np.sqrt(252)
    f['vol_63']   = r.rolling(63).std().shift(1) * np.sqrt(252)
    f['vol_ratio']= (r.rolling(10).std() / r.rolling(63).std()).shift(1)
    # MA cross
    f['ma_10_50'] = (p.rolling(10).mean() / p.rolling(50).mean() - 1).shift(1)
    f['ma_50_200']= (p.rolling(50).mean() / p.rolling(200).mean() - 1).shift(1)
    # 52-week high proximity (George & Hwang 2004)
    f['nearness_52w'] = (p / p.rolling(252).max()).shift(1)
    # Bollinger Band z-score
    f['bb_z'] = ((p - p.rolling(20).mean()) /
                  p.rolling(20).std()).shift(1)
    # Forward return (target)
    f['fwd_5d'] = r.shift(-SIGNAL_HOR).rolling(SIGNAL_HOR).sum() - cost * 2
    return f.replace([np.inf, -np.inf], np.nan)

all_feats = {t: build_features(prices[t], log_ret[t])
             for t in TICKERS if t in prices.columns}
print(f'Features per ticker: {list(all_feats.values())[0].shape[1]-1} signals')

## 3. Individual Signal Tests (IC / ICIR)

In [ ]:
signal_cols = ['mom_5d','mom_21d','mom_63d','mom_126d','mom_252d',
               'rev_1d','rev_5d','rsi','vol_21','vol_63','vol_ratio',
               'ma_10_50','ma_50_200','nearness_52w','bb_z']

results = []
for ticker, feat in all_feats.items():
    df = feat.dropna(subset=['fwd_5d'])
    for sig in signal_cols:
        if sig not in df: continue
        s = df[[sig, 'fwd_5d']].dropna()
        if len(s) < 100: continue
        ic    = s[sig].corr(s['fwd_5d'])
        n     = len(s)
        se    = np.sqrt((1 - ic**2) / (n - 2)) if n > 2 else np.nan
        t_ic  = ic / se if se and se > 0 else np.nan
        icir  = ic / s[sig].rolling(63).corr(s['fwd_5d']).std()                 if s[sig].rolling(63).corr(s['fwd_5d']).std() > 0 else np.nan
        results.append({'ticker': ticker, 'signal': sig,
                        'IC': round(ic, 5), 't_IC': round(t_ic, 3),
                        'N': n, 'ICIR': round(float(icir), 3) if icir else np.nan})

res_df = pd.DataFrame(results)
top    = res_df.dropna().sort_values('t_IC', ascending=False).head(15)
print('Top 15 signals by |t_IC|:')
print(top.to_string(index=False))
hlz_pass = res_df[res_df['t_IC'].abs() >= ALPHA_HLZ]
print(f'\nPass HLZ |t|≥{ALPHA_HLZ}: {len(hlz_pass)} / {len(res_df)}')

## 4. Fama-MacBeth Cross-Sectional Regression

In [ ]:
# Panel: stack all tickers by date
panel_rows = []
for ticker, feat in all_feats.items():
    df = feat[signal_cols + ['fwd_5d']].dropna()
    df['ticker'] = ticker
    panel_rows.append(df)
panel = pd.concat(panel_rows).sort_index()

# Rolling 252-day Fama-MacBeth: cross-sectional regression each day
gammas = {s: [] for s in signal_cols}
dates_fm = sorted(set(panel.index))

for dt in dates_fm[252:]:
    cross = panel.loc[dt].dropna(subset=['fwd_5d'])
    if len(cross) < 4: continue
    y = cross['fwd_5d'].values
    for sig in signal_cols:
        if sig not in cross: continue
        x = cross[sig].values
        mask = ~(np.isnan(x) | np.isnan(y))
        if mask.sum() < 4: continue
        slope = np.polyfit(x[mask], y[mask], 1)[0]
        gammas[sig].append(slope)

print('Fama-MacBeth gamma estimates (mean ± SE):')
fm_results = []
for sig, g in gammas.items():
    if len(g) < 30: continue
    arr   = np.array(g)
    mean  = arr.mean()
    se    = arr.std() / np.sqrt(len(arr))
    t     = mean / se if se > 0 else np.nan
    fm_results.append({'signal': sig, 'mean_gamma': round(mean,6),
                       'SE': round(se,6), 't_stat': round(t,3),
                       'N_months': len(arr),
                       'significant': abs(t) >= 1.96 if t else False})
fm_df = pd.DataFrame(fm_results).sort_values('t_stat', key=abs, ascending=False)
print(fm_df.to_string(index=False))
sig_fm = fm_df[fm_df['significant']]
print(f'\nSignificant (|t|≥1.96): {len(sig_fm)} / {len(fm_df)}')

## 5. XGBoost Walk-Forward

In [ ]:
TRAIN_W, TEST_W = 504, 63

# Use the combined panel — stack ticker features cross-sectionally
all_X, all_y = [], []
for ticker, feat in all_feats.items():
    df = feat[signal_cols + ['fwd_5d']].dropna()
    all_X.append(df[signal_cols])
    all_y.append(df['fwd_5d'])

X_all = pd.concat(all_X).sort_index()
y_all = pd.concat(all_y).sort_index()
common = X_all.index.intersection(y_all.index)
X_all, y_all = X_all.loc[common], y_all.loc[common]

n = len(X_all)
sc = StandardScaler()
preds, actuals = [], []

for start in range(TRAIN_W, n - TEST_W, TEST_W):
    Xtr = X_all.iloc[start-TRAIN_W:start]
    ytr = y_all.iloc[start-TRAIN_W:start]
    Xte = X_all.iloc[start:start+TEST_W]
    yte = y_all.iloc[start:start+TEST_W]
    mask = ytr.notna() & Xtr.notna().all(axis=1)
    if mask.sum() < 50: continue
    Xts = sc.fit_transform(Xtr[mask]); Xes = sc.transform(Xte.fillna(0))
    m = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.7,
                          random_state=42, verbosity=0)
    m.fit(Xts, ytr[mask])
    preds.extend(m.predict(Xes).tolist())
    actuals.extend(yte.tolist())

preds, actuals = np.array(preds), np.array(actuals)
direction_acc = ((preds > 0) == (actuals > 0)).mean()
ret_strategy  = actuals * np.sign(preds)
ann_ret  = ret_strategy.mean() * 252
ann_vol  = ret_strategy.std()  * np.sqrt(252)
sharpe   = (ann_ret - RF_ANNUAL) / ann_vol if ann_vol > 0 else 0

print(f'OOS direction accuracy : {direction_acc:.3f}')
print(f'OOS Ann. return        : {ann_ret:.4f} ({ann_ret:.2%})')
print(f'OOS Sharpe ratio       : {sharpe:.4f}')
print(f'OOS observations       : {len(preds)}')

## 6. Feature Importance (SHAP)

In [ ]:
try:
    import shap
    # Fit on last training fold for SHAP analysis
    Xtr_last = X_all.iloc[-TRAIN_W:].fillna(0)
    ytr_last  = y_all.iloc[-TRAIN_W:]
    Xts_last  = sc.fit_transform(Xtr_last)
    m_shap    = xgb.XGBRegressor(n_estimators=200, max_depth=3,
                                   learning_rate=0.05, random_state=42, verbosity=0)
    m_shap.fit(Xts_last, ytr_last)
    explainer  = shap.TreeExplainer(m_shap)
    shap_vals  = explainer.shap_values(Xts_last[:200])
    shap.summary_plot(shap_vals, Xtr_last[:200], feature_names=signal_cols,
                      show=False, max_display=12)
    plt.title('XGBoost SHAP Feature Importance — Signal Predictability (RQ1)',
              fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/figures/03_shap_importance.png', dpi=150)
    plt.show(); print('SHAP plot saved')
except ImportError:
    print('pip install shap  to generate SHAP plots')
    # Fallback: built-in XGBoost importance
    fi = pd.Series(m.feature_importances_, index=signal_cols).sort_values()
    fi.plot(kind='barh', color='#1A9988', figsize=(8,6))
    plt.title('XGBoost Feature Importance (RQ1)', fontweight='bold')
    plt.tight_layout(); plt.savefig('../results/figures/03_feature_importance.png', dpi=150)
    plt.show()